# Build `oil_network` schema

Defines the full relational schema for the asset-centric oil-network model in PostgreSQL, schema `oil_network`.

**Repeatable:** the first cell drops and recreates the schema, so the entire notebook can be re-run from scratch with no leftover state. Inspect the result in DBeaver (right-click the schema → ER Diagram).

**Built incrementally**, one concept at a time.

- **Step 1** — `locations`, `assets`, `graphs`, `nodes`. Real-world identity, where things are, how assets are positioned in a graph.
- **Step 2** — `commodities`, `variable_types`, `node_types`, `variables`. The variable layer (slots) plus the node-type catalog.
- **Step 3** — `time_series`, `time_series_data`. Asset-level series with vintaged values.
- **Step 4** — `scenarios`. Per-graph scenario registry.
- **Step 5** — `variable_assignments`, `node_type_default_formulas`, plus two `BEFORE INSERT/UPDATE` triggers enforcing the same-graph constraints that Postgres CHECK can't express.

## 0. Init

In [1]:
import os
import pandas as pd
from sqlalchemy import create_engine, text
from IPython.display import display

PG_HOST   = os.environ.get("PG_HOST",   "localhost")
PG_PORT   = os.environ.get("PG_PORT",   "5432")
PG_DB     = os.environ.get("PG_DB",     "eia_crude")
PG_USER   = os.environ.get("PG_USER",   "eia_user")
PG_PASS   = os.environ.get("PG_PASS",   "eia_password")
PG_SCHEMA = os.environ.get("PG_SCHEMA", "oil_network")
PG_URL    = f"postgresql+psycopg2://{PG_USER}:{PG_PASS}@{PG_HOST}:{PG_PORT}/{PG_DB}"

engine = create_engine(
    PG_URL,
    connect_args={"options": f"-csearch_path={PG_SCHEMA},public"},
    future=True,
)

with engine.connect() as c:
    ver, db, usr = c.execute(text("SELECT version(), current_database(), current_user")).one()
print(f"Connected:     {db} as {usr}")
print(f"Server:        {ver.splitlines()[0]}")
print(f"Target schema: {PG_SCHEMA}")

Connected:     eia_crude as eia_user
Server:        PostgreSQL 18.3 on x86_64-windows, compiled by msvc-19.44.35223, 64-bit
Target schema: oil_network


## 1. Reset schema

Drop and recreate `oil_network` so this notebook is idempotent — it can be re-run end-to-end and always converges to the same state.

In [2]:
with engine.begin() as c:
    c.execute(text(f"DROP SCHEMA IF EXISTS {PG_SCHEMA} CASCADE"))
    c.execute(text(f"CREATE SCHEMA {PG_SCHEMA}"))

print(f"Schema `{PG_SCHEMA}` reset.")

Schema `oil_network` reset.


## Step 1 — assets, locations, graphs, nodes

Four tables that together let us state "this asset, located here, is represented by this node in this graph."

**`locations`** — physical place. Common geographic fields are flat columns (`lat`, `lon`, `country`, `state`, `padd`, `county`, `sea`) for direct querying; everything else (e.g. coastline, basin label, climate zone) goes in `attributes` JSONB. `location_id` is an arbitrary stable text key.

**`assets`** — real-world identity. An asset has a name, optional description, a nullable `location_id` FK, and a JSONB `attributes` blob for anything else (capacity_bpd, operator, Nelson Index, sub-class metadata). Time series will eventually attach here, not to nodes — one asset, one TS, reusable across graphs.

**`graphs`** — a named asset graph. Lets us model the same assets in multiple ways (coarse vs fine, crude vs products, version 1 vs version 2).

**`nodes`** — junction. Each row is "asset X appears in graph Y as node N." `UNIQUE(graph_id, asset_id)` ensures one node per asset per graph; `node_id` is globally unique so variables and edges can reference it with a single column. No `role` column yet (deferred until we have a concrete use).

In [3]:
with engine.begin() as c:
    c.execute(text("""
        CREATE TABLE locations (
            location_id TEXT PRIMARY KEY,
            name        TEXT,
            lat         DOUBLE PRECISION,
            lon         DOUBLE PRECISION,
            country     TEXT,
            state       TEXT,
            padd        TEXT,
            county      TEXT,
            sea         TEXT,
            attributes  JSONB NOT NULL DEFAULT '{}'::jsonb
        )
    """))
    c.execute(text("CREATE INDEX ix_locations_state ON locations(state)"))
    c.execute(text("CREATE INDEX ix_locations_padd  ON locations(padd)"))

    c.execute(text("""
        CREATE TABLE assets (
            asset_id    TEXT PRIMARY KEY,
            name        TEXT NOT NULL,
            description TEXT,
            location_id TEXT REFERENCES locations(location_id) ON DELETE SET NULL,
            attributes  JSONB NOT NULL DEFAULT '{}'::jsonb,
            created_at  TIMESTAMPTZ NOT NULL DEFAULT now()
        )
    """))
    c.execute(text("CREATE INDEX ix_assets_location ON assets(location_id)"))

    c.execute(text("""
        CREATE TABLE graphs (
            graph_id    TEXT PRIMARY KEY,
            name        TEXT NOT NULL,
            description TEXT,
            attributes  JSONB NOT NULL DEFAULT '{}'::jsonb,
            created_at  TIMESTAMPTZ NOT NULL DEFAULT now()
        )
    """))

    c.execute(text("""
        CREATE TABLE nodes (
            node_id    TEXT PRIMARY KEY,
            graph_id   TEXT NOT NULL REFERENCES graphs(graph_id) ON DELETE CASCADE,
            asset_id   TEXT NOT NULL REFERENCES assets(asset_id),
            notes      TEXT,
            attributes JSONB NOT NULL DEFAULT '{}'::jsonb,
            UNIQUE(graph_id, asset_id)
        )
    """))
    c.execute(text("CREATE INDEX ix_nodes_graph ON nodes(graph_id)"))
    c.execute(text("CREATE INDEX ix_nodes_asset ON nodes(asset_id)"))

print("Step 1 tables created: locations, assets, graphs, nodes.")

Step 1 tables created: locations, assets, graphs, nodes.


## Step 2 — commodities, variable_types, node_types, variables

Adds the variable layer plus a `node_types` catalog so step 3's per-node-type formula defaults have a place to attach.

**`commodities`** — small lookup table. Crude, gasoline, WTI, Brent, etc. FK from `variables.commodity` (and later from `time_series.commodity`). Catches typos like `crude` vs `Crude`.

**`variable_types`** — six labels: `production`, `consumption`, `inventory`, `balancing_item`, `inflow`, `outflow`. Pure labels — no `balance_sign`, no special role for `balancing_item`. The mass-balance equation lives as a per-node formula in step 3.

**`node_types`** — small lookup. `nodes.node_type` is a nullable FK; a node without a type just doesn't pick up any default formulas, and every variable assignment must be explicit.

**`variables`** — slot definitions. One row per `(variable_type, commodity, node_id, related_node_id?)`. Two CHECKs:
- `inflow` and `outflow` must have a `related_node_id`; the other four types must not.
- `related_node_id` must differ from `node_id` (no self-edges).

Two partial UNIQUE indexes (same trick as before) handle uniqueness correctly across the nullable `related_node_id` column.

**Deferred to step 3 or later:**
- Same-graph constraint between `node_id` and `related_node_id` — needs a trigger (Postgres CHECK can't subquery cross-row).
- `node_type_default_formulas` table — pulls in step 3 alongside `variable_assignments`.

In [4]:
with engine.begin() as c:
    c.execute(text("""
        CREATE TABLE commodities (
            commodity   TEXT PRIMARY KEY,
            description TEXT,
            attributes  JSONB NOT NULL DEFAULT '{}'::jsonb
        )
    """))

    c.execute(text("""
        CREATE TABLE variable_types (
            variable_type TEXT PRIMARY KEY,
            description   TEXT,
            attributes    JSONB NOT NULL DEFAULT '{}'::jsonb
        )
    """))

    c.execute(text("""
        CREATE TABLE node_types (
            node_type   TEXT PRIMARY KEY,
            description TEXT,
            attributes  JSONB NOT NULL DEFAULT '{}'::jsonb
        )
    """))

    # Add nullable node_type FK to nodes.
    c.execute(text("""
        ALTER TABLE nodes
        ADD COLUMN node_type TEXT REFERENCES node_types(node_type)
    """))
    c.execute(text("CREATE INDEX ix_nodes_type ON nodes(node_type)"))

    c.execute(text("""
        CREATE TABLE variables (
            variable_id     TEXT PRIMARY KEY,
            variable_type   TEXT NOT NULL REFERENCES variable_types(variable_type),
            commodity       TEXT NOT NULL REFERENCES commodities(commodity),
            node_id         TEXT NOT NULL REFERENCES nodes(node_id) ON DELETE CASCADE,
            related_node_id TEXT          REFERENCES nodes(node_id) ON DELETE CASCADE,
            description     TEXT,
            attributes      JSONB NOT NULL DEFAULT '{}'::jsonb,
            CHECK (related_node_id IS NULL OR related_node_id <> node_id),
            CHECK (
                (variable_type IN ('inflow','outflow') AND related_node_id IS NOT NULL)
                OR
                (variable_type NOT IN ('inflow','outflow') AND related_node_id IS NULL)
            )
        )
    """))
    c.execute(text("""
        CREATE UNIQUE INDEX variables_uniq_nonrelational
            ON variables(variable_type, commodity, node_id)
            WHERE related_node_id IS NULL
    """))
    c.execute(text("""
        CREATE UNIQUE INDEX variables_uniq_relational
            ON variables(variable_type, commodity, node_id, related_node_id)
            WHERE related_node_id IS NOT NULL
    """))
    c.execute(text("CREATE INDEX ix_variables_node     ON variables(node_id)"))
    c.execute(text("CREATE INDEX ix_variables_ref_node ON variables(related_node_id)"))
    c.execute(text("CREATE INDEX ix_variables_type     ON variables(variable_type)"))

print("Step 2 tables created: commodities, variable_types, node_types, variables.")
print("Column added: nodes.node_type (nullable, FK to node_types).")

Step 2 tables created: commodities, variable_types, node_types, variables.
Column added: nodes.node_type (nullable, FK to node_types).


### Seed: variable_types

Six canonical types as plain labels — no signs, no special role for `balancing_item`. Mass-balance behaviour comes from per-node formulas in step 3.

In [5]:
VARIABLE_TYPE_SEED = [
    ("production",     "Volume produced (supply injected) at the node."),
    ("consumption",    "Volume consumed at the node."),
    ("inventory",      "Stock / inventory level at the node."),
    ("balancing_item", "Residual term that absorbs reporting discrepancies. "
                       "Defined per-node via formula in step 3."),
    ("inflow",         "Per-edge inflow into this node from related_node_id."),
    ("outflow",        "Per-edge outflow from this node to related_node_id."),
]

with engine.begin() as c:
    c.execute(
        text("INSERT INTO variable_types(variable_type, description) VALUES (:t, :d)"),
        [dict(t=t, d=d) for t, d in VARIABLE_TYPE_SEED],
    )

with engine.connect() as c:
    display(pd.read_sql(text("SELECT * FROM variable_types ORDER BY variable_type"), c))

,variable_type,description,attributes
0,balancing_item,Residual term that absorbs reporting discrepan...,{}
1,consumption,Volume consumed at the node.,{}
2,inflow,Per-edge inflow into this node from related_no...,{}
3,inventory,Stock / inventory level at the node.,{}
4,outflow,Per-edge outflow from this node to related_nod...,{}
5,production,Volume produced (supply injected) at the node.,{}


## Step 3 — time series

A time series describes one measurement over time. Its subject is an **asset** (`asset_id`, nullable). Assets are the single identity layer in the model: physical things (refineries, pipelines, basins) and abstract entities (PADD 5, regulatory aggregates, benchmarks) all live in `assets`. PADD 5 is an asset whose `attributes` JSONB can flag it as an aggregation rather than a facility.

**`time_series_types`** — small lookup classifying *the measurement* (`production`, `consumption`, etc.). Different vocabulary from `variable_types`: TS types describe what the data measures; variable types describe a slot's role in the mass balance. Same labels at first, free to diverge later (TS could grow `forecast`, `target`, `actual`).

**`time_series`** — the catalog. `commodity` and `time_series_type` describe the measurement; `asset_id` describes the subject; `source` records who published it.

**`time_series_data`** — the actual values. PK is `(timeseries_id, observation_date, saved_date)`, so a value revision lands as a *new row* with a later `saved_date` rather than overwriting history. Step-forward semantics: a value at `t` holds until the next observation. All values are mbd.

The link from a TS to a variable on a node happens later, in step 5's `variable_assignments`.

In [6]:
with engine.begin() as c:
    c.execute(text("""
        CREATE TABLE time_series_types (
            time_series_type TEXT PRIMARY KEY,
            description      TEXT,
            attributes       JSONB NOT NULL DEFAULT '{}'::jsonb
        )
    """))

    c.execute(text("""
        CREATE TABLE time_series (
            timeseries_id    TEXT PRIMARY KEY,
            name             TEXT NOT NULL,
            description      TEXT,
            time_series_type TEXT          REFERENCES time_series_types(time_series_type),
            commodity        TEXT NOT NULL REFERENCES commodities(commodity),
            asset_id         TEXT          REFERENCES assets(asset_id) ON DELETE SET NULL,
            source           TEXT NOT NULL,
            attributes       JSONB NOT NULL DEFAULT '{}'::jsonb,
            created_at       TIMESTAMPTZ NOT NULL DEFAULT now()
        )
    """))
    c.execute(text("CREATE INDEX ix_ts_asset     ON time_series(asset_id)"))
    c.execute(text("CREATE INDEX ix_ts_commodity ON time_series(commodity)"))
    c.execute(text("CREATE INDEX ix_ts_type      ON time_series(time_series_type)"))
    c.execute(text("CREATE INDEX ix_ts_source    ON time_series(source)"))

    c.execute(text("""
        CREATE TABLE time_series_data (
            timeseries_id    TEXT NOT NULL REFERENCES time_series(timeseries_id) ON DELETE CASCADE,
            observation_date DATE NOT NULL,
            saved_date       DATE NOT NULL,
            value            DOUBLE PRECISION,
            PRIMARY KEY(timeseries_id, observation_date, saved_date)
        )
    """))
    c.execute(text("CREATE INDEX ix_tsd_obsdate ON time_series_data(observation_date)"))
    c.execute(text("CREATE INDEX ix_tsd_saved   ON time_series_data(saved_date)"))

    # Seed time_series_types with the same six labels as variable_types.
    # The two tables are conceptually distinct and free to diverge.
    c.execute(
        text("INSERT INTO time_series_types(time_series_type, description) VALUES (:t, :d)"),
        [
            {"t": "production",     "d": "Measurement of volume produced."},
            {"t": "consumption",    "d": "Measurement of volume consumed."},
            {"t": "inventory",      "d": "Measurement of stock / inventory level."},
            {"t": "balancing_item", "d": "Measurement of a balance residual."},
            {"t": "inflow",         "d": "Measurement of per-edge inflow."},
            {"t": "outflow",        "d": "Measurement of per-edge outflow."},
        ],
    )

print("Step 3 tables created: time_series_types, time_series, time_series_data.")
print("Seeded time_series_types with 6 labels.")

Step 3 tables created: time_series_types, time_series, time_series_data.
Seeded time_series_types with 6 labels.


## Step 4 — scenarios

A *scenario* is a particular instantiation of a graph: which variables are assigned what data, with what formulas, over what date range. One scenario lives over one graph. We can have several scenarios on the same graph (for sensitivity analysis, for testing alternate balance closures) and several scenarios across graphs (a "coarse" scenario on the coarse graph and a "fine" scenario on the fine graph).

Step 4 just registers scenarios. The actual variable→TS-or-formula assignments happen in step 5.

In [7]:
with engine.begin() as c:
    c.execute(text("""
        CREATE TABLE scenarios (
            scenario_id TEXT PRIMARY KEY,
            graph_id    TEXT NOT NULL REFERENCES graphs(graph_id) ON DELETE CASCADE,
            name        TEXT NOT NULL,
            description TEXT,
            attributes  JSONB NOT NULL DEFAULT '{}'::jsonb,
            created_at  TIMESTAMPTZ NOT NULL DEFAULT now()
        )
    """))
    c.execute(text("CREATE INDEX ix_scenarios_graph ON scenarios(graph_id)"))

print("Step 4 table created: scenarios.")

Step 4 table created: scenarios.


## Step 5 — variable_assignments, default formulas, same-graph triggers

This is the layer that turns variables into something the evaluator can compute.

**`variable_assignments`** — for `(scenario_id, variable_id, effective_from)`, says either:
- `timeseries_id` set → values come from that time series, **or**
- `formula` set → values are computed per timestep from `formula`, with `formula_inputs[]` listing the variable_ids the formula reads.

A single CHECK enforces exactly one of the two is populated: `num_nonnulls(timeseries_id, formula) = 1`. No enum kind column — the populated column *is* the kind. "Zero" is just `formula = '0'`.

`effective_from` is in the PK so a single variable can switch sources mid-run (e.g. a TS until 2020 and a formula afterwards).

**`node_type_default_formulas`** — defaults keyed on `(node_type, variable_type, commodity?)`. When a node has a `node_type` and a variable doesn't have an explicit assignment in a scenario, this is what fills in. `commodity` is nullable: NULL means "applies to every commodity at this node type." Two partial unique indexes preserve the at-most-one rule for both the NULL and non-NULL cases.

**Two triggers** for cross-row constraints Postgres CHECKs can't express:

1. `variables` → `related_node_id` must be in the same graph as `node_id`.
2. `variable_assignments` → the variable's node must live in the scenario's graph.

Both fire `BEFORE INSERT OR UPDATE`, look up the relevant `graph_id`s, and raise on mismatch.

In [8]:
with engine.begin() as c:
    c.execute(text("""
        CREATE TABLE variable_assignments (
            scenario_id     TEXT NOT NULL REFERENCES scenarios(scenario_id) ON DELETE CASCADE,
            variable_id     TEXT NOT NULL REFERENCES variables(variable_id) ON DELETE CASCADE,
            effective_from  DATE NOT NULL,
            timeseries_id   TEXT REFERENCES time_series(timeseries_id) ON DELETE RESTRICT,
            formula         TEXT,
            formula_inputs  TEXT[] NOT NULL DEFAULT '{}'::text[],
            notes           TEXT,
            PRIMARY KEY(scenario_id, variable_id, effective_from),
            CHECK (num_nonnulls(timeseries_id, formula) = 1)
        )
    """))
    c.execute(text("CREATE INDEX ix_va_variable ON variable_assignments(variable_id)"))
    c.execute(text("CREATE INDEX ix_va_ts       ON variable_assignments(timeseries_id)"))
    c.execute(text("CREATE INDEX ix_va_scenario ON variable_assignments(scenario_id)"))

    c.execute(text("""
        CREATE TABLE node_type_default_formulas (
            node_type        TEXT NOT NULL REFERENCES node_types(node_type) ON DELETE CASCADE,
            variable_type    TEXT NOT NULL REFERENCES variable_types(variable_type),
            commodity        TEXT          REFERENCES commodities(commodity),
            default_formula  TEXT NOT NULL,
            description      TEXT
        )
    """))
    c.execute(text("""
        CREATE UNIQUE INDEX node_type_defaults_uniq_with_commodity
            ON node_type_default_formulas(node_type, variable_type, commodity)
            WHERE commodity IS NOT NULL
    """))
    c.execute(text("""
        CREATE UNIQUE INDEX node_type_defaults_uniq_no_commodity
            ON node_type_default_formulas(node_type, variable_type)
            WHERE commodity IS NULL
    """))

    # Trigger 1: variables.related_node_id must share a graph with variables.node_id.
    c.execute(text("""
        CREATE OR REPLACE FUNCTION variables_check_same_graph()
        RETURNS TRIGGER AS $$
        DECLARE
            n_graph TEXT;
            r_graph TEXT;
        BEGIN
            IF NEW.related_node_id IS NULL THEN
                RETURN NEW;
            END IF;
            SELECT graph_id INTO n_graph FROM nodes WHERE node_id = NEW.node_id;
            SELECT graph_id INTO r_graph FROM nodes WHERE node_id = NEW.related_node_id;
            IF n_graph IS DISTINCT FROM r_graph THEN
                RAISE EXCEPTION
                    'variables.node_id (%) is in graph % but related_node_id (%) is in graph %',
                    NEW.node_id, n_graph, NEW.related_node_id, r_graph;
            END IF;
            RETURN NEW;
        END;
        $$ LANGUAGE plpgsql
    """))
    c.execute(text("""
        CREATE TRIGGER variables_check_same_graph_trg
        BEFORE INSERT OR UPDATE ON variables
        FOR EACH ROW EXECUTE FUNCTION variables_check_same_graph()
    """))

    # Trigger 2: variable_assignments.variable_id must live on a node in the
    # scenario's graph.
    c.execute(text("""
        CREATE OR REPLACE FUNCTION variable_assignments_check_graph()
        RETURNS TRIGGER AS $$
        DECLARE
            s_graph TEXT;
            v_graph TEXT;
        BEGIN
            SELECT graph_id INTO s_graph
            FROM scenarios WHERE scenario_id = NEW.scenario_id;
            SELECT n.graph_id INTO v_graph
            FROM variables v JOIN nodes n ON n.node_id = v.node_id
            WHERE v.variable_id = NEW.variable_id;
            IF s_graph IS DISTINCT FROM v_graph THEN
                RAISE EXCEPTION
                    'scenario % is over graph %, but variable % is on a node in graph %',
                    NEW.scenario_id, s_graph, NEW.variable_id, v_graph;
            END IF;
            RETURN NEW;
        END;
        $$ LANGUAGE plpgsql
    """))
    c.execute(text("""
        CREATE TRIGGER variable_assignments_check_graph_trg
        BEFORE INSERT OR UPDATE ON variable_assignments
        FOR EACH ROW EXECUTE FUNCTION variable_assignments_check_graph()
    """))

print("Step 5 created: variable_assignments, node_type_default_formulas, plus 2 same-graph triggers.")

Step 5 created: variable_assignments, node_type_default_formulas, plus 2 same-graph triggers.


## Verify

List the tables and their column shapes — a quick sanity check while DBeaver is open on the side.

In [9]:
with engine.connect() as c:
    tables = pd.read_sql(text("""
        SELECT table_name
        FROM information_schema.tables
        WHERE table_schema = :s
        ORDER BY table_name
    """), c, params={"s": PG_SCHEMA})
    columns = pd.read_sql(text("""
        SELECT table_name, column_name, data_type, is_nullable, column_default
        FROM information_schema.columns
        WHERE table_schema = :s
        ORDER BY table_name, ordinal_position
    """), c, params={"s": PG_SCHEMA})
    fks = pd.read_sql(text("""
        SELECT tc.table_name, kcu.column_name,
               ccu.table_name  AS references_table,
               ccu.column_name AS references_column
        FROM information_schema.table_constraints tc
        JOIN information_schema.key_column_usage kcu
          ON tc.constraint_name = kcu.constraint_name
         AND tc.table_schema    = kcu.table_schema
        JOIN information_schema.constraint_column_usage ccu
          ON ccu.constraint_name = tc.constraint_name
         AND ccu.table_schema    = tc.table_schema
        WHERE tc.constraint_type = 'FOREIGN KEY'
          AND tc.table_schema = :s
        ORDER BY tc.table_name, kcu.column_name
    """), c, params={"s": PG_SCHEMA})

print(f"Tables in {PG_SCHEMA}:")
display(tables)
print("Columns:")
display(columns)
print("Foreign keys:")
display(fks)

Tables in oil_network:


,table_name
0,assets
1,commodities
2,graphs
3,locations
4,node_type_default_formulas
5,node_types
6,nodes
7,scenarios
8,time_series
9,time_series_data


Columns:


,table_name,column_name,data_type,is_nullable,column_default
0,assets,asset_id,text,NO,NaN
1,assets,name,text,NO,NaN
2,assets,description,text,YES,NaN
3,assets,location_id,text,YES,NaN
4,assets,attributes,jsonb,NO,'{}'::jsonb
...,...,...,...,...,...
72,variables,commodity,text,NO,NaN
73,variables,node_id,text,NO,NaN
74,variables,related_node_id,text,YES,NaN
75,variables,description,text,YES,NaN


Foreign keys:


,table_name,column_name,references_table,references_column
0,assets,location_id,locations,location_id
1,node_type_default_formulas,commodity,commodities,commodity
2,node_type_default_formulas,node_type,node_types,node_type
3,node_type_default_formulas,variable_type,variable_types,variable_type
4,nodes,asset_id,assets,asset_id
5,nodes,graph_id,graphs,graph_id
6,nodes,node_type,node_types,node_type
7,scenarios,graph_id,graphs,graph_id
8,time_series,asset_id,assets,asset_id
9,time_series,commodity,commodities,commodity
